# HW-2 on Kaggle
Same code as `hw-2.ipynb`, with Kaggle paths. The competition data is attached, so nothing is downloaded and no token is needed.

Settings before running: **Accelerator: GPU T4 x2**, **Internet: on**, and the competition added under **Input**. Use **Save Version → Save & Run All**, so the run continues on Kaggle after the browser closes. Files are kept only in the output of a saved version.

In [ ]:
%pip install -q ultralytics

In [ ]:
import os
import shutil
import sys

COMPETITION = 'multiple-object-video-detection-hw-2-ucu-ai-2026'
IN_COLAB = False
TRAIN = True
RUNS = '/kaggle/working/runs'
os.makedirs(RUNS, exist_ok=True)

candidates = [f'/kaggle/input/{COMPETITION}', f'/kaggle/input/competitions/{COMPETITION}']
path = next((p for p in candidates if os.path.exists(f'{p}/test_images.csv')), None)
assert path, f'Competition data not found, /kaggle/input has: {os.listdir("/kaggle/input")}'

!nvidia-smi --query-gpu=name,memory.total --format=csv
print('Data:', path)
print('Free disk, GB:', round(shutil.disk_usage('/kaggle/working').free / 1e9))

## Data overview
Drone video clips from VisDrone with 8 object classes. Train and val have labels, test has only frames. Everything below follows the same order as hw-1: look at the data, find what is wrong with it, then put the fixes into one `clean()`.

In [ ]:
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
COLORS = {'train': '#2a78d6', 'val': '#eb6834', 'test': '#1baf7a'}


def draw_boxes(image, boxes, color):
    thickness = max(2, image.shape[1] // 700)
    for x, y, w, h in boxes[['x', 'y', 'w', 'h']].to_numpy(int):
        cv2.rectangle(image, (x, y), (x + w, y + h), color, thickness)
    return image

In [ ]:
DATA = Path(path)


def load_split(split):
    coco = json.loads((DATA / split / 'annotations_coco.json').read_text())
    images = pd.DataFrame(coco['images']).rename(columns={'id': 'image_id'})
    boxes = pd.DataFrame(coco['annotations'])
    boxes[['x', 'y', 'w', 'h']] = np.array(boxes['bbox'].tolist())
    boxes = boxes.drop(columns='bbox').merge(
        images[['image_id', 'video', 'frame_id', 'width', 'height']], on='image_id')
    return images, boxes


train_images, train_boxes = load_split('train')
val_images, val_boxes = load_split('val')
test_images = pd.read_csv(DATA / 'test_images.csv')

print('Train:', train_images.shape, train_boxes.shape)
print('Val:', val_images.shape, val_boxes.shape)
print('Test:', test_images.shape)
display(train_boxes.head())

In [ ]:
print(train_boxes.isna().sum())
print(val_boxes.isna().sum())
print(test_images.isna().sum())

In [ ]:
dd = pd.DataFrame({
    'column': ['x', 'y', 'w', 'h', 'category_id', 'orig_category_id', 'iscrowd',
               'truncation', 'occlusion', 'track_id', 'video', 'frame_id'],
    'description': ['left edge, px', 'top edge, px', 'box width, px', 'box height, px',
                    'merged class, 1-8', 'original VisDrone class', '1 = ignore region, not an object',
                    '1 = object cut by the frame edge', '0 none, 1 partial, 2 heavy',
                    'same object across frames', 'clip name', 'frame number in the clip'],
}).set_index('column')
dd['dtype'] = train_boxes.dtypes.astype(str)
dd['missing_%'] = (train_boxes.isna().mean() * 100).round(2)
dd[['dtype', 'missing_%', 'description']]

Before any statistics, look at real frames. Green boxes are labelled objects, red boxes are `iscrowd=1` rows.

In [ ]:
clips = train_images.video.drop_duplicates().sample(6, random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(16, 6.5))

for ax, clip in zip(axes.flatten(), clips):
    frame = train_images[train_images.video == clip].iloc[0]
    boxes = train_boxes[train_boxes.image_id == frame.image_id]
    image = cv2.cvtColor(cv2.imread(str(DATA / 'train' / frame.file_name)), cv2.COLOR_BGR2RGB)
    draw_boxes(image, boxes[boxes.iscrowd == 0], (0, 220, 0))
    draw_boxes(image, boxes[boxes.iscrowd == 1], (230, 40, 40))
    ax.imshow(image)
    ax.set_title(f'{clip} ({frame.width}x{frame.height})', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

### What are the `iscrowd` rows?
`track_id`, `truncation` and `occlusion` are missing on the same rows, about a third of the table. Check how those rows split by class.

In [ ]:
pd.crosstab(train_boxes.iscrowd, train_boxes.category_id)

Every class has exactly the same number of `iscrowd=1` rows. Real objects cannot do that, so these look like the same regions copied once per class. Count the unique regions.

In [ ]:
crowd = train_boxes[train_boxes.iscrowd == 1]
regions = crowd.drop_duplicates(['image_id', 'x', 'y', 'w', 'h'])

print('rows:', len(crowd), ' unique regions:', len(regions), ' copies:', len(crowd) / len(regions))
print(crowd.orig_category_id.value_counts())

Confirmed: 63,843 regions, each repeated 8 times. They come from the original VisDrone classes 0 (ignored region) and 11 (others). So non-target objects and areas the annotators skipped are both stored as `iscrowd=1`, and neither is a training target. The other original classes are merged into the 8 target classes (for example pedestrian and people become person).

Do the YOLO `.txt` labels contain them? Compare the number of lines with the real boxes on every frame.

In [ ]:
real_per_frame = train_boxes[train_boxes.iscrowd == 0].groupby('image_id').size()
expected = train_images.image_id.map(real_per_frame).fillna(0).astype(int)

lines = [len((DATA / 'train' / 'labels' / r.video / f'{r.frame_id:07d}.txt').read_text().splitlines())
         for r in tqdm(train_images.itertuples(), total=len(train_images))]

print('frames where labels differ from real boxes:', (np.array(lines) != expected).sum())
print('frames without any object:', (expected == 0).sum())

The labels already skip the ignore regions, so they can be used as they are. Next, check the real boxes the same way hw-1 checked for impossible values.

In [ ]:
def add_geometry(df):
    df = df[df.iscrowd == 0].copy()
    df['rel_size'] = np.sqrt(df.w * df.h / (df.width * df.height))
    df['cx'] = (df.x + df.w / 2) / df.width
    df['cy'] = (df.y + df.h / 2) / df.height
    df['class_name'] = df.category_id.map(lambda i: CLASSES[i - 1])
    return df


real_train = add_geometry(train_boxes)
real_val = add_geometry(val_boxes)

for name, df in [('train', real_train), ('val', real_val)]:
    outside = (df.x < 0) | (df.y < 0) | (df.x + df.w > df.width) | (df.y + df.h > df.height)
    print(name)
    print('  non-positive size:', ((df.w <= 0) | (df.h <= 0)).sum())
    print('  outside the frame:', outside.sum())
    print('  duplicate track in a frame:', df.duplicated(['video', 'frame_id', 'track_id']).sum())
    print('  exact duplicate boxes:', df.duplicated(['image_id', 'category_id', 'x', 'y', 'w', 'h']).sum())
    print('  smallest side, px:', np.minimum(df.w, df.h).min())

There is nothing to fix here: unlike hw-1, the real labels have no impossible values, and even the smallest boxes are valid objects.

### Box size
Drone footage has very small objects, and this decides the input size of the model. Compare sizes as a share of the frame, because frame resolutions differ.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
columns = [('w', 'box width, px'), ('h', 'box height, px'), ('rel_size', 'sqrt(box area / frame area)')]

for ax, (col, title) in zip(axes, columns):
    both = pd.concat([real_train[col], real_val[col]])
    bins = np.geomspace(both.quantile(0.001), both.quantile(0.999), 40)
    ax.hist(real_train[col], bins=bins, density=True, histtype='step', lw=2, color=COLORS['train'], label='train')
    ax.hist(real_val[col], bins=bins, density=True, histtype='step', lw=2, color=COLORS['val'], label='val')
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_yticks([])
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
order = real_train.groupby('class_name').rel_size.median().sort_values().index
sns.boxplot(data=real_train, x='rel_size', y='class_name', order=order, showfliers=False,
            color=COLORS['train'], log_scale=True, ax=ax)
ax.set_xlabel('sqrt(box area / frame area)')
ax.set_ylabel('')
ax.set_title('Box size by class', loc='left')
plt.tight_layout()
plt.show()

for imgsz in (640, 960, 1280):
    side = np.sqrt(real_train.w * real_train.h) * imgsz / np.maximum(real_train.width, real_train.height)
    print(f'imgsz {imgsz}: {(side < 8).mean():.1%} of boxes under 8 px, {(side < 16).mean():.1%} under 16 px')

Train and val have almost the same size distribution, so val is a fair check for object size. Objects are tiny: at `imgsz=640`, 54.2% of boxes are under 16 px, compared with 24.0% at 960 and 9.8% at 1280. Person, bicycle and motor are the smallest classes, and bus is the largest.

Decision: the baseline uses `imgsz=960`, and 1280 is an early experiment. Anchor sizes for the anchor-based models should come from these distributions.

### Classes

In [ ]:
share = pd.DataFrame({'train': real_train.class_name.value_counts(normalize=True),
                      'val': real_val.class_name.value_counts(normalize=True)}).sort_values('train')

fig, ax = plt.subplots(figsize=(9, 4.5))
share.plot.barh(ax=ax, color=[COLORS['train'], COLORS['val']], width=0.8)
ax.set_xlabel('share of boxes')
ax.set_ylabel('')
ax.set_title('Class share in train and val', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(real_train.class_name.value_counts())
print('car / bus:', round((real_train.class_name == 'car').sum() / (real_train.class_name == 'bus').sum(), 1))

In [ ]:
occluded = real_train.groupby('class_name').occlusion.apply(lambda s: (s > 0).mean()).sort_values()

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(occluded.index, occluded.values, color=COLORS['train'])
for y, v in enumerate(occluded.values):
    ax.text(v, y, f' {v:.0%}', va='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_title('Share of train boxes that are partly or heavily occluded', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

Car and person make up about 75% of train boxes, and there are 52 cars for every bus. Val has the opposite order (person 44%, car 28%), and bus has only 264 val boxes, so its val AP will be noisy. Results should be read per class, not only as one number.

Between 43% and 65% of the boxes in every class are partly or heavily occluded, with truck the highest. Scenes are crowded, so an aggressive NMS could remove real neighbouring objects.

### Where objects are and how many

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist2d(real_train.cx, real_train.cy, bins=60, range=[[0, 1], [0, 1]], cmap='Blues')
axes[0].invert_yaxis()
axes[0].set_title('Box centres in the frame (train)')
axes[0].set_xlabel('x / width')
axes[0].set_ylabel('y / height')

per_frame = {'train': real_train.groupby('image_id').size(), 'val': real_val.groupby('image_id').size()}
bins = np.arange(0, 301, 10)
for name, counts in per_frame.items():
    axes[1].hist(counts, bins=bins, density=True, histtype='step', lw=2, color=COLORS[name], label=name)
axes[1].set_title('Objects per frame')
axes[1].set_xlabel('objects')
axes[1].set_yticks([])
axes[1].legend()

plt.tight_layout()
plt.show()

print(per_frame['train'].describe().round(1))

Objects gather around the centre and upper middle of the frame, because the camera looks down at an angle. A horizontal flip keeps this geometry, a vertical flip does not, so vertical flips stay off.

The median train frame has 43 objects and the densest has 180. That is below the 300 queries RT-DETR uses by default and the 500 detections scored per image, so neither limit should cut real objects. Val has more sparse frames than train, with a clear peak at 10-20 objects.

### How much clips differ
A dataset average can hide large differences between sequences, so repeat the main statistics for each clip.

In [ ]:
per_clip = real_train.groupby('video').agg(boxes=('id', 'size'), frames=('image_id', 'nunique'),
                                           box_size=('rel_size', 'median'))
per_clip['objects_per_frame'] = per_clip.boxes / per_clip.frames
per_clip['person_share'] = real_train.class_name.eq('person').groupby(real_train.video).mean()

columns = [('objects_per_frame', 'Objects per frame'), ('box_size', 'Median box size'),
           ('person_share', 'Share of person boxes')]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, (col, title) in zip(axes, columns):
    values = per_clip[col].sort_values().to_numpy()
    ax.bar(range(len(values)), values, width=0.8, color=COLORS['train'])
    ax.set_title(title, loc='left')
    ax.set_xlabel('clips, sorted')
    ax.set_xticks([])
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

per_clip[[c for c, _ in columns]].describe().loc[['min', '50%', 'max']].round(3)

Clips are very different. Objects per frame go from 5 to 143 (median 44), median box size from 0.019 to 0.061 of the frame, and the share of person boxes from 0% to 100%. This is why the splits are made by clip, and why one validation number depends so much on which clips are in val.

### Ignore regions
Objects inside a region are visible but not labelled, so a detector would be punished for finding them. The usual fix is to paint the regions out. Before doing that, check how much of the frame they cover and whether real boxes sit inside them.

In [ ]:
pairs = real_train[['id', 'image_id', 'x', 'y', 'w', 'h']].merge(
    regions[['image_id', 'x', 'y', 'w', 'h']], on='image_id', suffixes=('', '_r'))
overlap_w = (np.minimum(pairs.x + pairs.w, pairs.x_r + pairs.w_r) - np.maximum(pairs.x, pairs.x_r)).clip(lower=0)
overlap_h = (np.minimum(pairs.y + pairs.h, pairs.y_r + pairs.h_r) - np.maximum(pairs.y, pairs.y_r)).clip(lower=0)
pairs['covered'] = overlap_w * overlap_h / (pairs.w * pairs.h)
covered = pairs.groupby('id').covered.max()

coverage = (regions.w * regions.h / (regions.width * regions.height)).groupby(regions.image_id).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(coverage.clip(upper=0.3), bins=30, color=COLORS['train'])
axes[0].set_title('Share of the frame inside ignore regions')
axes[0].set_xlabel('share of frame')
axes[0].set_ylabel('frames')
axes[1].hist(covered[covered > 0], bins=20, color=COLORS['train'])
axes[1].set_title('Real boxes touching a region: how much is covered')
axes[1].set_xlabel('share of the box inside a region')
axes[1].set_ylabel('boxes')
plt.tight_layout()
plt.show()

print('frames with regions:', coverage.size, 'of', len(train_images))
print('mean frame coverage:', round(coverage.mean(), 4))
print('real boxes touching a region:', (covered > 0).sum())
print('real boxes more than half covered:', (covered > 0.5).sum())

Regions appear on 18,090 of 24,201 frames (75%), but they cover only about 3.2% of a frame on average, so painting them removes little. However, 38,801 real boxes touch a region and 11,733 are more than half inside one. The peak at 1.0 shows many labelled objects sit fully inside a region.

Decision: paint the regions, then copy every labelled box back. Whether masking helps at all is worth testing later against training on the original frames.

### Train vs test
hw-1 compared train and test feature by feature. Test has no labels here, so compare what is available: resolution, clip length and the images themselves.

In [ ]:
def resolution_share(df):
    return (df.width.astype(str) + 'x' + df.height.astype(str)).value_counts(normalize=True)


resolutions = pd.DataFrame({'train': resolution_share(train_images), 'val': resolution_share(val_images),
                            'test': resolution_share(test_images)}).fillna(0)
resolutions = resolutions.sort_values('test', ascending=False)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(resolutions, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=False, ax=ax)
ax.set_title('Share of frames at each resolution', loc='left')
plt.tight_layout()
plt.show()

In [ ]:
lengths = {'train': train_images.groupby('video').size(), 'val': val_images.groupby('video').size(),
           'test': test_images.groupby('clip').size()}


def image_stats(files, n=300):
    rows = []
    for f in pd.Series(files).sample(n, random_state=42):
        gray = cv2.imread(str(DATA / f), cv2.IMREAD_GRAYSCALE)
        rows.append({'brightness': gray.mean(), 'contrast': gray.std()})
    return pd.DataFrame(rows)


stats = {'train': image_stats('train/' + train_images.file_name),
         'val': image_stats('val/' + val_images.file_name),
         'test': image_stats(test_images.file_name)}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
bins = np.geomspace(30, 1500, 18)
for name, values in lengths.items():
    axes[0].hist(values, bins=bins, histtype='step', lw=2, color=COLORS[name], label=name)
axes[0].set_xscale('log')
axes[0].set_title('Frames per clip')
axes[0].set_ylabel('clips')
axes[0].legend()

for ax, col in zip(axes[1:], ['brightness', 'contrast']):
    both = pd.concat([s[col] for s in stats.values()])
    bins = np.linspace(both.min(), both.max(), 25)
    for name, s in stats.items():
        ax.hist(s[col], bins=bins, density=True, histtype='step', lw=2, color=COLORS[name], label=name)
    ax.set_title(col)
    ax.set_yticks([])
    ax.legend()

plt.tight_layout()
plt.show()

print(pd.DataFrame({k: v.describe() for k, v in lengths.items()}).loc[['count', 'min', '50%', 'max']])
print(pd.DataFrame({k: v.mean() for k, v in stats.items()}).round(1))

- **Resolution:** only 1344x756 and 1920x1080 appear in both train and test. Test frames are never larger than 1920x1080, and about 16% are low resolution (680x382 or 960x540), while train goes up to 2720x1530. Frames are not resized on disk, because the model input size handles this.
- **Val:** 34% of val frames are 3840x2160, so one kind of footage has a large effect on val scores.
- **Clip length:** test clips are short (35-148 frames) compared with train (58-1,424).
- **Images:** on 300 sampled frames per split, test is darker (mean brightness 101.6 vs 120.5) and has lower contrast (42.5 vs 47.5). Brightness augmentation should help. Val forms separate groups because it has only 7 clips.

### How similar are neighbouring frames?
hw-1 had independent rows. Video frames are not independent, so check how fast a clip changes: the mean pixel difference between a frame and the frame `gap` steps later.

In [ ]:
def small_gray(file_name):
    gray = cv2.imread(str(DATA / 'train' / file_name), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(gray, (320, 180)).astype(float)


gaps = [1, 2, 5, 10, 30, 60]
long_clips = train_images.groupby('video').size().loc[lambda s: s >= 200].index
rows = []

for clip in pd.Series(long_clips).sample(8, random_state=42):
    files = train_images[train_images.video == clip].sort_values('frame_id').file_name.tolist()
    for start in np.linspace(0, len(files) - max(gaps) - 1, 4).astype(int):
        first = small_gray(files[start])
        for gap in gaps:
            rows.append({'clip': clip, 'gap': gap, 'diff': np.abs(first - small_gray(files[start + gap])).mean()})

change = pd.DataFrame(rows).groupby('gap')['diff'].median()

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(change.index, change.values, marker='o', lw=2, color=COLORS['train'])
ax.set_xscale('log')
ax.set_xticks(gaps, labels=gaps)
ax.set_ylim(0)
ax.set_xlabel('gap, frames')
ax.set_ylabel('mean pixel difference (0-255)')
ax.set_title('Frames change slowly inside a clip', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(change.round(1))

The difference is 6.6 at a gap of 1, 15.8 at a gap of 5 and 29.5 at a gap of 60 (8 clips, 4 start points each). A gap of 5 already reaches over half of the change seen at 60, so every 5th frame keeps most of the variety for a fifth of the training cost. This is why `TRAIN_STRIDE = 5`.

The stride does not balance the clips: a long clip still gives more frames than a short one. Taking a fixed number of frames per clip is a separate experiment.

## Data cleaning: `clean()`
One deterministic function for train and val. It works row by row and learns nothing from the data, so there is no leakage between folds.
- `iscrowd=1`: removed from the targets, one copy of each region kept as `ignore`;
- class ids: shifted from 1-8 to the 0-7 used in the YOLO labels.

In [ ]:
def clean(boxes: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    real = boxes[boxes.iscrowd == 0].copy()
    real['cls'] = real.category_id - 1
    ignore = boxes[boxes.iscrowd == 1].drop_duplicates(['image_id', 'x', 'y', 'w', 'h'])
    return real, ignore


train_clean, train_ignore = clean(train_boxes)
val_clean, val_ignore = clean(val_boxes)

print('Train:', len(train_clean), 'boxes,', len(train_ignore), 'ignore regions')
print('Val:', len(val_clean), 'boxes,', len(val_ignore), 'ignore regions')

## Splits by clip
hw-1 used `KFold(5, shuffle=True, random_state=42)`. Here a random split would put almost identical frames on both sides, so `GroupKFold` keeps each clip in one fold. The provided train and val share no clips.

In [ ]:
assert not set(train_images.video) & set(val_images.video)

gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
train_images['fold'] = -1
for k, (_, rows) in enumerate(gkf.split(train_images, groups=train_images.video)):
    train_images.loc[train_images.index[rows], 'fold'] = k

display(train_images.groupby('fold').agg(clips=('video', 'nunique'), frames=('image_id', 'size')))

folds = real_train.assign(fold=real_train.image_id.map(train_images.set_index('image_id').fold))
fold_share = pd.crosstab(folds.fold, folds.class_name, normalize='index')[CLASSES]

fig, ax = plt.subplots(figsize=(9, 3.4))
sns.heatmap(fold_share, annot=True, fmt='.2f', cmap='Blues', cbar=False, ax=ax)
ax.set_title('Class share in each fold', loc='left')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

Folds differ a lot: car share goes from 33% to 55%, and fold size from 4,002 to 6,057 frames. A single split depends heavily on which clips it contains.

Five folds per detector do not fit the GPU budget, so detector comparisons use the provided val split with fixed settings, and the folds are kept for checking the final pipeline.

In [ ]:
TRAIN_STRIDE = 5

train_subset = train_images[train_images.groupby('video').cumcount() % TRAIN_STRIDE == 0]
print('Train frames:', len(train_images), '->', len(train_subset))

subset_boxes = real_train[real_train.image_id.isin(train_subset.image_id)]
compare = pd.DataFrame({'all frames': real_train.class_name.value_counts(normalize=True),
                        'every 5th frame': subset_boxes.class_name.value_counts(normalize=True)}).sort_values('all frames')
compare['diff'] = compare['all frames'] - compare['every 5th frame']
print(compare.round(4))

Class shares change by at most 0.04 percentage points, so the subset keeps the class balance.

## Build the training folder
Ignore regions are painted grey, then every labelled box is copied back so no real object is lost. Val is painted too, otherwise detections of unlabelled objects would count as false positives. The competition does not penalise detections inside ignore regions, but test has no ignore files and stays as it is, so val scores are only a guide to the leaderboard.

Frames without regions are symlinks to the originals. The folder is on the Colab local disk, so a new runtime has to rebuild it.

In [ ]:
PREP = Path('/content/prep' if IN_COLAB else 'prep')
MASK_VALUE = 114


def mask_frame(image, regions, objects):
    masked = image.copy()
    for x, y, w, h in regions:
        masked[y:y + h, x:x + w] = MASK_VALUE
    for x, y, w, h in objects:
        masked[y:y + h, x:x + w] = image[y:y + h, x:x + w]
    return masked


def boxes_by_image(boxes):
    return {k: v.to_numpy(int) for k, v in boxes.groupby('image_id')[['x', 'y', 'w', 'h']]}


def build_split(split, images, real, ignore):
    shutil.rmtree(PREP / split, ignore_errors=True)
    regions, objects = boxes_by_image(ignore), boxes_by_image(real)
    no_boxes = np.empty((0, 4), int)

    for r in tqdm(images.itertuples(), total=len(images), desc=split):
        label = Path(r.file_name).relative_to('images').with_suffix('.txt')
        image_path = PREP / split / r.file_name
        label_path = PREP / split / 'labels' / label
        image_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.parent.mkdir(parents=True, exist_ok=True)

        source = DATA / split / r.file_name
        if r.image_id in regions:
            image = cv2.imread(str(source))
            masked = mask_frame(image, regions[r.image_id], objects.get(r.image_id, no_boxes))
            cv2.imwrite(str(image_path), masked, [cv2.IMWRITE_JPEG_QUALITY, 95])
        else:
            image_path.symlink_to(source)
        shutil.copy(DATA / split / 'labels' / label, label_path)


build_split('train', train_subset, train_clean, train_ignore)
build_split('val', val_images, val_clean, val_ignore)

data_yaml = PREP / 'data.yaml'
data_yaml.write_text(f"path: {PREP.resolve()}\ntrain: train/images\nval: val/images\n"
                     f"nc: {len(CLASSES)}\nnames: {json.dumps(CLASSES)}\n")
print(data_yaml.read_text())

Check that nothing was lost: every frame has an image and a label file, and the number of boxes matches `clean()`.

In [ ]:
for split, images, real in [('train', train_subset, train_clean), ('val', val_images, val_clean)]:
    n_images = len(list((PREP / split / 'images').rglob('*.jpg')))
    labels = list((PREP / split / 'labels').rglob('*.txt'))
    n_boxes = sum(len(p.read_text().splitlines()) for p in labels)
    expected = real.image_id.isin(images.image_id).sum()
    print(f'{split}: {n_images} images, {len(labels)} labels, {n_boxes} boxes (expected {expected})')
    assert n_images == len(labels) == len(images) and n_boxes == expected

In [ ]:
area = train_ignore[train_ignore.image_id.isin(train_subset.image_id)]
area = (area.w * area.h / (area.width * area.height)).groupby(area.image_id).sum()
frame = train_subset[train_subset.image_id == area.idxmax()].iloc[0]
boxes = train_clean[train_clean.image_id == frame.image_id]
regions_in_frame = train_ignore[train_ignore.image_id == frame.image_id]

before = cv2.cvtColor(cv2.imread(str(DATA / 'train' / frame.file_name)), cv2.COLOR_BGR2RGB)
after = cv2.cvtColor(cv2.imread(str(PREP / 'train' / frame.file_name)), cv2.COLOR_BGR2RGB)
draw_boxes(before, regions_in_frame, (230, 40, 40))
draw_boxes(after, boxes, (0, 220, 0))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(before)
axes[0].set_title('Before: ignore regions in red')
axes[1].imshow(after)
axes[1].set_title('After: regions painted grey, labelled boxes in green')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Resizing, normalization and augmentation
- **Resizing:** frames stay at their original size on disk. Ultralytics letterboxes them to `imgsz` during training and prediction, keeping the aspect ratio.
- **Normalization:** pixel values are divided by 255 inside the framework.
- **Augmentation:** Ultralytics defaults for the baseline: mosaic (turned off for the last 10 epochs), horizontal flip with probability 0.5, scale 0.5, translate 0.1, and HSV jitter (`hsv_v=0.4` covers the brightness gap with test). `flipud` stays 0, as explained above.
- **Reproducibility:** `seed=42` for training, `random_state=42` for sampling, `TRAIN_STRIDE` and `IMGSZ` are set in the notebook.

## Preprocessing summary

| hw-1 | hw-2 | Result |
|---|---|---|
| Missing values | `isna()` on all tables | Missing only on the `iscrowd=1` rows |
| Impossible values (`attendance_rate` > 100) | Box size, frame bounds, duplicates | None in real boxes |
| Systematic error (4x `study_time`) | `iscrowd=1` rows | Ignore regions copied 8 times, removed from targets |
| `clean()` | `clean()` | Same function for train and val |
| Imputation, one-hot encoding | Not needed | Boxes have no missing values and classes are already numbers |
| Train vs test comparison | Resolution, clip length, brightness | Different resolutions; test clips are shorter and darker |
| `KFold` | `GroupKFold` by clip | No clip on both sides of a split |
| 100k subsample | `TRAIN_STRIDE = 5` | 24,201 -> 4,868 frames |

`PREP / 'data.yaml'` is the file to pass to the model.

## Baseline model
The assignment recommends a complete simple pipeline first, so this section goes from training to a valid submission. `yolo11n` is a small anchor-free one-stage detector that trains quickly on a T4. It only proves the pipeline works; the tuned detector comparisons come later.

Weights are saved to Drive. `last.pt` keeps the optimizer until training ends, so `unfinished()` can tell an interrupted run from a finished one: an interrupted run continues from its last epoch, and a finished run is not trained again. With `TRAIN = False` the cell only loads the saved weights, and it stops with an error if they come from an unfinished run.

After a kernel restart, run the setup cells and `b_imports`, then continue from here. The training folder stays on the runtime, so the data section does not have to run again.

In [ ]:
import gc

import torch
from ultralytics import YOLO

DATA = Path(path)
PREP = Path('/content/prep' if IN_COLAB else 'prep')
data_yaml = PREP / 'data.yaml'
test_images = pd.read_csv(DATA / 'test_images.csv')

assert data_yaml.exists(), 'No training folder: run the section "Build the training folder" first'

In [ ]:
IMGSZ = 960
EPOCHS = 20
MODEL = 'yolo11n.pt'
RUN_NAME = 'baseline-yolo11n-960'

run_dir = Path(RUNS) / RUN_NAME
last_weights, best_weights = run_dir / 'weights' / 'last.pt', run_dir / 'weights' / 'best.pt'


def unfinished(last):
    return torch.load(last, map_location='cpu', weights_only=False).get('optimizer') is not None


if TRAIN:
    if not last_weights.exists():
        YOLO(MODEL).train(data=str(data_yaml), imgsz=IMGSZ, epochs=EPOCHS, batch=-1, seed=42,
                          project=RUNS, name=RUN_NAME, exist_ok=True)
    elif unfinished(last_weights):
        YOLO(str(last_weights)).train(resume=True)

assert not unfinished(last_weights), 'Training is not finished: set TRAIN = True and run this cell again'

model = YOLO(best_weights)
print('Weights:', best_weights)
print('Epochs logged:', len(pd.read_csv(run_dir / 'results.csv')))

### Validation
`mAP50-95` and `mAP50` are the standard COCO-style averages, and speed is the inference time per frame. These numbers use the masked val frames. `AP_small`, which the detector sections must report, needs a COCO evaluator that is still to be added.

In [ ]:
MAX_DET = 500

metrics = model.val(data=str(data_yaml), imgsz=IMGSZ, max_det=MAX_DET, plots=False, verbose=False)

display(pd.Series({'mAP50-95': metrics.box.map, 'mAP50': metrics.box.map50, 'precision': metrics.box.mp,
                   'recall': metrics.box.mr, 'ms per frame': metrics.speed['inference']}).round(3))

class_map = pd.Series(0.0, index=CLASSES)
class_map.iloc[metrics.box.ap_class_index] = metrics.box.ap
class_map = class_map.sort_values()

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(class_map.index, class_map.values, color=COLORS['val'])
for y, v in enumerate(class_map.values):
    ax.text(v, y, f' {v:.2f}', va='center', fontsize=9)
ax.set_xlim(0, max(class_map.max() * 1.2, 0.1))
ax.set_title('Baseline mAP50-95 by class on val', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## Submission
One row per test image in `test_images.csv`. `PredictionString` is a list of `class score x y width height` groups, with (x, y) as the top-left corner in pixels of the original image. This is not the normalised YOLO format. Images with no detections stay empty, and at most 500 detections per image are scored.

A low `CONF` keeps uncertain boxes, because mAP uses the whole ranking of detections rather than one threshold.

Predictions are saved to Drive one clip at a time, so if the kernel stops, running the cell again skips the finished clips. If the weights change, delete the `predictions` folder of that run first.

In [ ]:
CONF = 0.01
parts = run_dir / 'predictions'
parts.mkdir(exist_ok=True)


def to_prediction_string(result):
    rows = zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.int().tolist())
    return ' '.join(f'{c} {s:.4f} {x1:.1f} {y1:.1f} {x2 - x1:.1f} {y2 - y1:.1f}'
                    for (x1, y1, x2, y2), s, c in rows)


for clip, frames in tqdm(test_images.groupby('clip')):
    part = parts / f'{clip}.json'
    if part.exists():
        continue
    paths = [str(DATA / f) for f in frames.file_name]
    results = model.predict(paths, stream=True, imgsz=IMGSZ, conf=CONF, max_det=MAX_DET, verbose=False)
    part.write_text(json.dumps({i: to_prediction_string(r) for i, r in zip(frames.image_id, results)}))
    gc.collect()

predictions = {}
for part in parts.glob('*.json'):
    predictions.update(json.loads(part.read_text()))

submission = pd.DataFrame({'image_id': test_images.image_id,
                           'PredictionString': test_images.image_id.map(predictions)})
submission.head()

Check the file before using a submission: every test image appears once, each detection has 6 values, classes are 0-7, and boxes stay inside their frame.

In [ ]:
values = submission.PredictionString.str.split()
counts = values.str.len() // 6

assert len(submission) == len(test_images) and submission.image_id.is_unique
assert (values.str.len() % 6 == 0).all() and counts.max() <= MAX_DET

detections = pd.DataFrame(
    [[i, *group] for i, row in zip(submission.image_id, values)
     for group in np.array(row, dtype=float).reshape(-1, 6)],
    columns=['image_id', 'cls', 'score', 'x', 'y', 'w', 'h']).merge(test_images, on='image_id')

assert detections.cls.between(0, len(CLASSES) - 1).all() and detections.score.between(0, 1).all()
assert (detections.w >= 0).all() and (detections.h >= 0).all()
assert (detections.x >= -1).all() and (detections.y >= -1).all()
assert (detections.x + detections.w <= detections.width + 1).all()
assert (detections.y + detections.h <= detections.height + 1).all()

print('images:', len(submission), ' detections:', len(detections), ' empty images:', (counts == 0).sum())
print(detections.cls.astype(int).map(lambda i: CLASSES[i]).value_counts())

In [ ]:
samples = test_images.drop_duplicates(['width', 'height']).sort_values('width').iloc[[0, -1]]
sure = detections[detections.score > 0.3]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, frame in zip(axes, samples.itertuples()):
    image = cv2.cvtColor(cv2.imread(str(DATA / frame.file_name)), cv2.COLOR_BGR2RGB)
    draw_boxes(image, sure[sure.image_id == frame.image_id], (0, 220, 0))
    ax.imshow(image)
    ax.set_title(f'{frame.clip} ({frame.width}x{frame.height}), score > 0.3')
    ax.axis('off')
plt.tight_layout()
plt.show()

The boxes above should sit on objects in both the smallest and the largest frame size. If they are shifted or scaled, the coordinates are wrong, so do not submit.

In [ ]:
submission.to_csv('submission.csv', index=False)
shutil.copy('submission.csv', Path(RUNS) / f'submission-{RUN_NAME}.csv')
print(Path('submission.csv').stat().st_size / 1e6, 'MB')

## Submit
When the saved version finishes, open it, check the boxes in the plot above, and submit `submission.csv` from the **Output** tab with **Submit to Competition**.